[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abhijeet1520/langgraph-workshop/blob/main/notebooks/level-1-basics/05_basic_agent.ipynb)

# Basic Agent

**Level 1** · Part of the [LangGraph Workshop](https://github.com/Abhijeet1520/langgraph-workshop).

> **Source:** migrated from `snehotosh/langgraph-notebooks` ([original](https://github.com/snehotosh/langgraph-notebooks/blob/main/4_BasicAgent.ipynb)) — see [PROVENANCE.md](https://github.com/Abhijeet1520/langgraph-workshop/blob/main/docs/PROVENANCE.md).

**Provider setup:** set `OPENAI_API_KEY` / `OPENAI_API_BASE` / `OPENAI_MODEL` first — [docs/setup/providers.md](https://github.com/Abhijeet1520/langgraph-workshop/blob/main/docs/setup/providers.md). The first cells install requirements and verify the connection.

## Goal:
- Build a Basic Langgraph Agent

In [ ]:
# --- Workshop provider config (edit ONLY these 3 values) ---------------------
import os

os.environ.setdefault("OPENAI_API_KEY", "")   # <- paste your key
os.environ.setdefault("OPENAI_API_BASE", "")  # <- paste your base URL
os.environ.setdefault("OPENAI_MODEL", "openai/gpt-4.1-mini")  # <- your model id

# Verify
for _v in ("OPENAI_API_KEY", "OPENAI_API_BASE", "OPENAI_MODEL"):
    print(_v, "=", "set" if os.getenv(_v) else "**MISSING — see docs/setup/providers.md**")
# -----------------------------------------------------------------------------

In [ ]:
from langchain_openai import ChatOpenAI
import os

def make_llm(**overrides):
    """Provider-agnostic LLM factory used across the workshop."""
    cfg = dict(
        model=os.environ["OPENAI_MODEL"],
        api_key=os.environ["OPENAI_API_KEY"],
        base_url=os.environ["OPENAI_API_BASE"],
        temperature=0,
    )
    cfg.update(overrides)
    return ChatOpenAI(**cfg)

llm = make_llm()

# Connection check (1 token)
resp = llm.invoke("Say OK")
print("✅ LLM reachable:", resp.content[:60])


#### imports

In [ ]:
from typing import TypedDict, Annotated, Sequence
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage, SystemMessage, BaseMessage
from langchain_core.tools import tool

from IPython.display import Image, display

import os
from dotenv import load_dotenv

In [ ]:
import pkg_resources
print(pkg_resources.get_distribution("langgraph").version)

In [ ]:
load_dotenv()

#### AgentState

- 👉 `add_messages`: Whenever a new message is added to the state, don’t overwrite "messages". Instead, append it to the existing list.
- 👉 `Sequence`: Type hint: "messages" is a sequence (list, tuple, etc.) of `BaseMessage` objects.
- 👉 `BaseMessage`: can be `HumanMessage`, `AIMessage`, `SystemMessage`, `ToolMessage`

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

#### Define tools

In [ ]:
@tool
def add(a:int, b:int):
    """ This is an addition function that adds two numbers"""
    return a + b

@tool
def sub(a:int, b:int):
    """ This is a subtraction function that subtracts two numbers"""
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """This is a multiplication function that multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

In [ ]:
tools = [add, sub, multiply]

#### Define Model

In [ ]:
model = make_llm().bind_tools(tools)

In [ ]:
def model_call(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content="You are my AI assistant, please answer my query to the best of your ability.")
    response = model.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}

In [ ]:
'''
'tool_calls': 
[{'id': 'call_yZ97kMZuNXrgpn7w1jOJwjRW', 
  'function': {'arguments': 
							'{"a":40,"b":12}', 
							'name': 'add'}, 
							'type': 'function'
			  }
]
'''
def should_continue(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1] ## Extracting the last AIMessage
    if not last_message.tool_calls:  
        return "end"
    else:
        return "continue"    

#### Building Graph

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("our_agent", model_call)
tool_node = ToolNode(tools=tools)
graph.add_node("tools", tool_node)
graph.set_entry_point("our_agent")
graph.add_conditional_edges("our_agent",
                            should_continue,
                            {
                                "continue": "tools",
                                "end": END
                            })
graph.add_edge("tools", "our_agent")
app = graph.compile()

#### Graph Visualization

In [ ]:
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
'''
param: stream is an iterator returned by app.stream(). 
So stream is not a list — it’s something you can iterate over (like reading a file line-by-line).
'''
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

#### Giving human input

In [ ]:
'''
[("user", "Add 40 + 12")]

This means:
It’s a list with one message (the user’s message).
Each message tuple has two parts:
    The role — "user" (or "system", "assistant", "tool", etc.)
    The content — the text of the message ("Add 40 + 12")
'''
inputs = {"messages" : [("user", "Add 40 + 12")]}

In [ ]:
print_stream(app.stream(input=inputs, stream_mode="values"))

In [ ]:
inputs2 = {"messages" : [("user", "Multiply that by 2.")]}
print_stream(app.stream(input=inputs2, stream_mode="values"))

**Note**: `No memory`